In [149]:
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import csv

In [150]:
class GridWorld:
    def __init__(self):
        self.gridworld = [[0] * 9 for i in range(6)]

        self.gridworld[1][2] = 2
        self.gridworld[2][2] = 2
        self.gridworld[3][2] = 2

        self.gridworld[0][7] = 2
        self.gridworld[1][7] = 2
        self.gridworld[2][7] = 2

        self.gridworld[4][5] = 2

        self.gridworld[2][0] = 1

        self.agent_location_x = 2
        self.agent_location_y = 0

    def action_space(self):
        return [0, 1, 2, 3]

    def step(self, a):
        if a == 0:
            self.agent_location_y_temp = max(self.agent_location_y - 1, 0)

            if self.gridworld[self.agent_location_x][self.agent_location_y_temp] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_y = self.agent_location_y_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 1:
            self.agent_location_x_temp = min(self.agent_location_x + 1, 5)

            if self.gridworld[self.agent_location_x_temp][self.agent_location_y] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_x = self.agent_location_x_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 2:
            self.agent_location_y_temp = min(self.agent_location_y + 1, 8)

            if self.gridworld[self.agent_location_x][self.agent_location_y_temp] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_y = self.agent_location_y_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        if a == 3:
            self.agent_location_x_temp = max(self.agent_location_x - 1, 0)

            if self.gridworld[self.agent_location_x_temp][self.agent_location_y] == 2:
                return self.agent_location_x, self.agent_location_y

            self.gridworld[self.agent_location_x][self.agent_location_y] = 0
            self.agent_location_x = self.agent_location_x_temp
            self.gridworld[self.agent_location_x][self.agent_location_y] = 1

        return self.agent_location_x, self.agent_location_y

    def reset(self):
        self.gridworld[1][2] = 2
        self.gridworld[2][2] = 2
        self.gridworld[3][2] = 2

        self.gridworld[0][7] = 2
        self.gridworld[1][7] = 2
        self.gridworld[2][7] = 2

        self.gridworld[4][5] = 2

        self.gridworld[2][0] = 1

        self.agent_location_x = 2
        self.agent_location_y = 0

In [151]:
env = GridWorld()


In [152]:
def one_hot_encoding(action):
    if action == 0:
        return [1, 0, 0, 0]
    elif action == 1:
        return [0, 1, 0, 0]
    elif action == 2:
        return [0, 0, 1, 0]
    elif action == 3:
        return [0, 0, 0, 1]

Gotta put in the inverted model sampling here.

In [153]:
def do_n_samples(n, inverse=False):

    data_X = []
    data_Y = []

    for i in range(n):
        previous_state = env.agent_location_x, env.agent_location_y

        action = np.random.randint(4)
        next_state = env.step(action)


        if inverse == True:
            input = list(previous_state) + list(next_state)

            output = one_hot_encoding(action)
        else:

          input = list(previous_state) + one_hot_encoding(action)

          output = list(next_state)

        data_X.append(input)

        data_Y.append(output)

    return data_X, data_Y

Writing out the program for intrinsic motivation.

Essentially, there are a few things here:

1. We need to compute an inverse dynamics model which is basically computes the action taken given a particular starting state and a particular ending state
2. We need to compute a model that would predict an particular ending state given a starting state and an action.

It need to be incredibly good at computing these two objectives. A good starting point is just a simple linear regression. We take the state, the action and the output needs to be the ending state.


A reasonable model for this would be a neural network with no activation function and two outputs. For the x and y of this.

I think the action could be better represented in a one hot encoding format than in a numerical format.

For the inverse dynamics model, the output could be coupled with a sigmoid function and a softmax function to produce probabilities of things.

The forward dynamics model is primarily a regression model and the inverse dynamics model is primarily a classification based neural network.

In [154]:
class ForwardDynamicsModel(nn.Module):
    def __init__(self):
        super(ForwardDynamicsModel, self).__init__()



        self.linear1 = nn.Linear(6, 100)
        self.linear2 = nn.Linear(100, 100)
        self.relu = nn.ReLU()
        self.linear3 = nn.Linear(100, 2)


        init.kaiming_normal_(self.linear1.weight, nonlinearity='relu')
        init.kaiming_normal_(self.linear2.weight, nonlinearity='relu')
        init.kaiming_normal_(self.linear3.weight, nonlinearity='relu')

    def forward(self, x):

        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.relu(x)
        y = self.linear3(x)
        return y

In [155]:
model_forward = ForwardDynamicsModel()
criterion_forward = nn.MSELoss()
optimizer_forward = torch.optim.SGD(model_forward.parameters(), lr=0.000001)

model_forward


ForwardDynamicsModel(
  (linear1): Linear(in_features=6, out_features=100, bias=True)
  (linear2): Linear(in_features=100, out_features=100, bias=True)
  (relu): ReLU()
  (linear3): Linear(in_features=100, out_features=2, bias=True)
)

In [208]:
# Training the model separately
for i in tqdm(range(3000)):

    data_X, data_Y = do_n_samples(2000)

    x = torch.tensor(data_X).float()
    y = torch.tensor(data_Y).float()

    predicted_output = model_forward(x)
    loss = criterion_forward(y, predicted_output)
    if i%100 == 0:
        print(loss)
    loss.backward()
    optimizer_forward.step()

  0%|          | 8/3000 [00:00<00:41, 71.62it/s]

tensor(11.6644, grad_fn=<MseLossBackward0>)


  4%|▎         | 109/3000 [00:01<00:50, 57.09it/s]

tensor(7.2574, grad_fn=<MseLossBackward0>)


  7%|▋         | 210/3000 [00:03<00:46, 60.29it/s]

tensor(5.0925, grad_fn=<MseLossBackward0>)


... (51 outputs left out)

tensor(0.9528, grad_fn=<MseLossBackward0>)


 97%|█████████▋| 2908/3000 [00:55<00:02, 40.75it/s]

tensor(1.4907, grad_fn=<MseLossBackward0>)


100%|██████████| 3000/3000 [00:58<00:00, 51.62it/s]


In [199]:
input, output = do_n_samples(5)
input = torch.tensor(input).float()
output = torch.tensor(output).float()

model_forward(input), output

(tensor([[2.4179, 4.4794],
         [2.9802, 4.1271],
         [3.5580, 3.2030],
         [2.4230, 3.6789],
         [1.4134, 3.6141]], grad_fn=<AddmmBackward0>),
 tensor([[2., 4.],
         [3., 4.],
         [3., 3.],
         [2., 3.],
         [1., 3.]]))

In [ ]:
# torch.save(model_forward.state_dict(), "forward_model_without_intrinsic.pth")

In [ ]:
# model_forward.load_state_dict(torch.load("forward_model_without_intrinsic.pth"))
# model_forward.eval()  # switch to inference mode

In [156]:
class InverseDynamicsModel(nn.Module):
    def __init__(self):
        super(InverseDynamicsModel, self).__init__()

        self.linear1 = nn.Linear(4, 100)
        self.linear5 = nn.Linear(100, 4)
        self.relu = nn.ReLU()

        self.batch_norm1 = nn.BatchNorm1d(100)
        self.batch_norm5 = nn.BatchNorm1d(4)

        init.kaiming_normal_(self.linear1.weight, nonlinearity='relu')
        init.kaiming_normal_(self.linear5.weight, nonlinearity='relu')


    def forward(self, x):

        x = self.linear1(x)
        x = self.batch_norm1(x)
        x = self.relu(x)
        x = self.linear5(x)
        x = self.batch_norm5(x)
        x = self.relu(x)

        x = F.softmax(x, dim=1)  # Softmax over the 4 outputs (per sample)

        return x

We also need to add intrinsic motivation to the mix. Essentially, intrinsic motivation would be the loss of the forward dynamics model. Essentially, it would be the difference in prediction for the next state and the actual next state. This would be multiplied by eta to compute intrinsic motivation.

In [157]:
model_inverse = InverseDynamicsModel()
criterion_inverse = nn.CrossEntropyLoss()
optimizer_inverse = torch.optim.Adam(model_inverse.parameters(), lr=0.001)

model_inverse

InverseDynamicsModel(
  (linear1): Linear(in_features=4, out_features=100, bias=True)
  (linear5): Linear(in_features=100, out_features=4, bias=True)
  (relu): ReLU()
  (batch_norm1): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batch_norm5): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [217]:
# Training the inverse dynamics model separately

for i in tqdm(range(6000)):

    data_X, data_Y = do_n_samples(2000, inverse=True)

    x = torch.tensor(data_X).float()
    y = torch.tensor(data_Y).float()

    predicted_output = model_inverse(x)
    loss = criterion_inverse(y, predicted_output)
    if i%500 == 0:
        print(loss)
    loss.backward()
    optimizer_inverse.step()

  0%|          | 8/6000 [00:00<03:33, 28.11it/s]

tensor(1.4922, grad_fn=<DivBackward1>)


  9%|▊         | 511/6000 [00:10<01:21, 67.61it/s]

tensor(1.0783, grad_fn=<DivBackward1>)


 17%|█▋        | 1009/6000 [00:20<01:24, 59.30it/s]

tensor(0.9130, grad_fn=<DivBackward1>)


... (15 outputs left out)

tensor(0.8666, grad_fn=<DivBackward1>)


 92%|█████████▏| 5511/6000 [01:47<00:07, 66.01it/s]

tensor(0.8315, grad_fn=<DivBackward1>)


100%|██████████| 6000/6000 [01:56<00:00, 51.32it/s]


In [184]:
input, output = do_n_samples(5, inverse=True)
input = torch.tensor(input).float()
output = torch.tensor(output).float()

input, model_inverse(input), output

(tensor([[3., 0., 4., 0.],
         [4., 0., 4., 1.],
         [4., 1., 4., 2.],
         [4., 2., 4., 1.],
         [4., 1., 4., 2.]]),
 tensor([[1.5124e-04, 9.9985e-01, 1.1789e-07, 4.2276e-08],
         [1.4457e-03, 9.1476e-05, 7.3425e-01, 2.6421e-01],
         [5.3037e-06, 3.4339e-03, 9.9655e-01, 1.2168e-05],
         [6.2504e-01, 1.9637e-07, 9.9500e-09, 3.7496e-01],
         [5.3037e-06, 3.4339e-03, 9.9655e-01, 1.2168e-05]],
        grad_fn=<SoftmaxBackward0>),
 tensor([[0., 1., 0., 0.],
         [0., 0., 1., 0.],
         [0., 0., 1., 0.],
         [1., 0., 0., 0.],
         [0., 0., 1., 0.]]))

In [121]:
input

tensor([[5., 2., 5., 3.],
        [5., 3., 5., 4.],
        [5., 4., 5., 5.],
        [5., 5., 5., 6.],
        [5., 6., 5., 7.]])

In [ ]:
# torch.save(model_inverse.state_dict(), "inverse_model_without_intrinsic.pth")

In [ ]:
# model_inverse.load_state_dict(torch.load("inverse_model_without_intrinsic.pth"))
# model_inverse.eval()  # switch to inference mode

Basically, we want to optimize things in a way that would find the parameters for lowering this particular loss.

The loss function would occur as follows:

-lambda * intrinsic_motivation_reward + (1-beta) * inverse_model_loss + (beta) * forward_model_loss

We would like to find the parameters that would minimize this loss.

Henceforth, the backpropogation must also happen in a similar way. It needs to combine the individual losses and intrinsic motivation and aim to find the derivatives that would minimize that loss function.

For starters we can try to train them separately and thenn combine them in some way.

What would the policy look like?

The policy should look to maximize intrinsic motivation.

The policy will be a NN similar to the inverse dynamics model. It will take in the current state and output the action. It will have sigmoidal function coupled with a softmax function..

The policy network's aim would be to maximize intrinsic motivation. Hence, it would be perhaps a gradient ascent style thing where the derivatives computed would be about the intrinsic motivation and trying to maximize that.

Let's start with something simple and train the forward dynamics model with a random policy to generate the data just for some practice and then we can scale things up from that.

In [162]:
policy = {}
for i in range(6):
  for j in range(9):
    policy[(i, j)] = [1, 1, 1, 1]

In [163]:

def select_action(policy, previous_state):

  previous_state_score = policy[previous_state]
  exp_scores = np.array(previous_state_score)
  probs = exp_scores / np.sum(exp_scores)
  action = np.random.choice(4, p=probs)

  return action

In [164]:
def get_n_samples_intrinsic_motivation(n, policy):

  data_X_forward = []
  data_Y_forward = []

  data_X_inverse = []
  data_Y_inverse = []

  data_policy = []
  for i in range(n):

    previous_state = env.agent_location_x, env.agent_location_y


    action = select_action(policy, previous_state)


    # action = np.random.randint(4)



    next_state = env.step(action)


    data_policy.append([previous_state, action])

    input = list(previous_state) + one_hot_encoding(action)
    output = list(next_state)

    data_X_forward.append(input)
    data_Y_forward.append(output)



    input = list(previous_state) + list(next_state)
    output = one_hot_encoding(action)
    data_X_inverse.append(input)
    data_Y_inverse.append(output)

  return data_X_forward, data_Y_forward, data_X_inverse, data_Y_inverse, data_policy


Here we have essentially used the policy network as a probability distribution where the policy network outputs the probabilities upon which the action is selected based upon that probability distirbution

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [166]:

with open('data_Y_forward.csv', 'a', newline='') as file:
  writer = csv.writer(file)


  for epoch in tqdm(range(6000)):


    data_X_forward, data_Y_forward, data_X_inverse, data_Y_inverse, data_policy = get_n_samples_intrinsic_motivation(2000, policy)

    writer.writerow(data_Y_forward)


    x_forward = torch.tensor(data_X_forward).float()
    y_forward = torch.tensor(data_Y_forward).float()


    x_inverse = torch.tensor(data_X_inverse).float()
    y_inverse = torch.tensor(data_Y_inverse).float()

    forward_output = model_forward(x_forward)
    forward_loss = criterion_forward(y_forward, forward_output)

    inverse_output = model_inverse(x_inverse)
    inverse_loss = criterion_inverse(y_inverse, inverse_output)




    mse_column = torch.mean((y_forward - forward_output) ** 2, dim=1, keepdim=True)

    eta = 3

    intrinsic_motivation_reward = eta * mse_column
    intrinsic_motivation_reward

    for i in range(2000):
      previous_state = data_policy[i][0]
      action = data_policy[i][1]

      policy[previous_state][action] = intrinsic_motivation_reward[i][0].item()



    if epoch % 100 == 0:
      print()
      print("Forward Loss:", forward_loss)
      print("Inverse Loss:", inverse_loss)


    optimizer_forward.zero_grad()
    optimizer_inverse.zero_grad()

    forward_loss.backward()
    optimizer_forward.step()

    inverse_loss.backward()
    optimizer_inverse.step()



  0%|          | 2/6000 [00:00<14:19,  6.98it/s]


Forward Loss: tensor(5.8375, grad_fn=<MseLossBackward0>)
Inverse Loss: tensor(1.2589, grad_fn=<DivBackward1>)


  2%|▏         | 102/6000 [00:11<14:57,  6.57it/s]


Forward Loss: tensor(6.4139, grad_fn=<MseLossBackward0>)
Inverse Loss: tensor(1.0970, grad_fn=<DivBackward1>)


  3%|▎         | 202/6000 [00:22<14:09,  6.83it/s]


Forward Loss: tensor(9.5685, grad_fn=<MseLossBackward0>)
Inverse Loss: tensor(1.0820, grad_fn=<DivBackward1>)


... (111 outputs left out)


Forward Loss: tensor(0.8146, grad_fn=<MseLossBackward0>)
Inverse Loss: tensor(0.8226, grad_fn=<DivBackward1>)


 98%|█████████▊| 5902/6000 [11:15<00:12,  7.82it/s]


Forward Loss: tensor(0.7750, grad_fn=<MseLossBackward0>)
Inverse Loss: tensor(0.8162, grad_fn=<DivBackward1>)


100%|██████████| 6000/6000 [11:26<00:00,  8.74it/s]


In [167]:
policy[(5, 8)]

[8.111165046691895, 4.418613910675049, 3.831587076187134, 3.7919769287109375]

In [142]:
previous_state_score = policy[(5, 8)]
exp_scores = np.exp(previous_state_score)
probs = exp_scores / np.sum(exp_scores)

probs

array([0.01297545, 0.00422798, 0.97974818, 0.00304839])

In [141]:
previous_state

(1, 8)

In [169]:
torch.save(model_inverse.state_dict(), "inverse_model.pth")
torch.save(model_forward.state_dict(), "forward_model.pth")
# torch.save(policy.state_dict(), "policy_model.pth")

There are two forms of data that we need to collect.
1. You want to collect the data for previous state, action -> next state
This is the forward dynamics model.
2. You want to collect the data for previous state, next state -> action
This is the inverse dynamics model.

Now, you also want to train the policy network.

The policy network would be basically be trained by the combined losses of forward, inverse and intrinsic motivaiton. Intrinsic motivation in this case would be loss of the forward dynamics model multiplied by eta.


You would use the data above to compute individual forward and inverse losses. Then, you would use the forward loss to compute intrinsic motivaiton.

You would then compute a combined loss.

Once, the combined loss is computed, this loss would then be used to train the three networks. Forward, inverse and policy

Here is how to think about the intrinsic motivation loss.

We are going to use the policy network to output the action based upon the large policy network and have it output actions based upon all that.

Then. use those actions to produce data and then compute the losses in the forward dynamics model and the inverse dynamics model. The forward dynamics model would essentially be the case that we are trying to determine how accurately can the agent detect the next state based upon the previous state and action. The inverse dynamics model is essentially given a previous state and next state, how accurately can you determine the action.

Intrinsic motivation is the eta times the loss of the forward dynamics model.

In [16]:
model_forward.load_state_dict(torch.load("forward_model.pth"))
model_inverse.load_state_dict(torch.load("inverse_model.pth"))
policy.load_state_dict(torch.load("policy_model.pth"))

<All keys matched successfully>

In [170]:
model_forward.eval()  # switch to inference mode
model_inverse.eval()  # switch to inference mode
# policy.eval()  # switch to inference mode

InverseDynamicsModel(
  (linear1): Linear(in_features=4, out_features=100, bias=True)
  (linear5): Linear(in_features=100, out_features=4, bias=True)
  (relu): ReLU()
  (batch_norm1): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batch_norm5): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [ ]:
policy_input = input[0][0:2].view(1, 2)

In [ ]:
policy(policy_input)

tensor([[0.2382, 0.2331, 0.2775, 0.2512]], grad_fn=<SoftmaxBackward0>)

In [197]:


input, output = do_n_samples(5)

In [198]:
input = torch.tensor(input).float()
output = torch.tensor(output).float()

In [199]:
input, output

(tensor([[2., 0., 0., 0., 1., 0.],
         [2., 1., 1., 0., 0., 0.],
         [2., 0., 0., 0., 1., 0.],
         [2., 1., 0., 1., 0., 0.],
         [3., 1., 0., 0., 0., 1.]]),
 tensor([[2., 1.],
         [2., 0.],
         [2., 1.],
         [3., 1.],
         [2., 1.]]))

In [200]:
predictions = model_forward(input)

predictions

tensor([[2.5096, 0.6631],
        [0.9764, 0.5574],
        [2.5096, 0.6631],
        [1.8319, 1.3871],
        [2.7739, 1.6400]], grad_fn=<AddmmBackward0>)

In [201]:
forward_loss = criterion_forward(output, predictions)

In [202]:
forward_loss

tensor(0.4628, grad_fn=<MseLossBackward0>)

In [203]:
inverse_loss = 1
beta = 0.35
(1-beta) * inverse_loss + (beta) * forward_loss

tensor(0.8120, grad_fn=<AddBackward0>)

In [225]:
input, output = do_n_samples(5, inverse=True)

In [226]:
input = torch.tensor(input).float()

In [227]:
model_inverse(input)

tensor([[7.2812e-05, 2.4024e-05, 9.9988e-01, 2.4024e-05],
        [1.5817e-04, 1.2889e-04, 3.4404e-04, 9.9937e-01],
        [4.7233e-04, 1.6743e-04, 3.7524e-04, 9.9898e-01],
        [9.9964e-01, 1.2863e-04, 1.1623e-04, 1.1146e-04],
        [6.3221e-04, 9.9916e-01, 1.0428e-04, 1.0667e-04]],
       grad_fn=<SoftmaxBackward0>)

In [229]:
output

[[0, 0, 1, 0], [0, 0, 0, 1], [0, 0, 0, 1], [1, 0, 0, 0], [0, 1, 0, 0]]

In [228]:
input

tensor([[2., 5., 2., 6.],
        [2., 6., 1., 6.],
        [1., 6., 0., 6.],
        [0., 6., 0., 5.],
        [0., 5., 1., 5.]])